# QPSK frequency acquisition before Costas tracking

This notebook slows down the new acquisition-range pass in the repo.

The bounded question is simple: **when does a Costas loop need help from a coarse frequency estimate, and how far does a 4th-power front end push the clean pull-in range before its own alias limit shows up?**


## 1. Why phase-only coarse correction runs out of room

A static phase estimate can rotate the constellation back toward the four corners at one instant.
That is useful, but it does nothing about the slope of the phase ramp.

If the received symbols look like

$$r_n = s_n e^{j(\\phi_0 + \\omega n)} + w_n,$$

then phase-only correction can remove `\\phi_0`, but the `\\omega n` term keeps accumulating. The loop has to eat that residual ramp by itself.


## 2. Why 4th-power acquisition works for QPSK

For Gray-style QPSK on the diagonals, the symbol contribution disappears under the 4th power up to a fixed phase factor.
That means

$$r_n^4 \\approx C e^{j4(\\phi_0 + \\omega n)},$$

so the phase difference between adjacent 4th-power samples carries `4\\omega`. Averaging those differences gives a coarse frequency estimate, and then the residual average phase gives a coarse phase estimate.


In [ ]:
from costaslab.analysis import sweep_acquisition_modes

rows = sweep_acquisition_modes([(-0.75 + 0.05 * idx) for idx in range(31)])
[(row.freq_offset, round(row.phase_only_tracked_rms_error, 3), round(row.freq_acquired_tracked_rms_error, 3)) for row in rows[:5]]


The useful signal in that table is not the tiny center offsets.
The interesting part is the outer region where phase-only correction starts failing while the 4th-power frequency estimate still hands the loop something nearly centered.


## 3. What the new range map is measuring

The artifact `assets/qpsk-acquisition-range-map.png` splits the problem three ways:

1. tracked RMS error versus true carrier offset
2. coarse estimated frequency versus true offset
3. a simple clean / marginal / failed regime bar for phase-only and frequency-assisted acquisition

That turns `the loop seems fine` into something measurable.


## 4. Caveat: the alias limit is part of the story

Because the coarse estimate comes from `e^{j4\\omega}`, it only identifies `\\omega` modulo `\\pi/2`.
A practical clean region therefore lives inside roughly

$$|\\omega| < \\pi/4,$$

before the frequency estimate wraps.

That is not a bug in the figure. It is the real scope boundary of this acquisition trick.


In [ ]:
best_outer = [row for row in rows if abs(row.freq_offset) >= 0.30]
[(row.freq_offset, round(row.coarse_frequency_estimate, 3)) for row in best_outer[:4]]


## 5. Problems and next moves

Try these next:

1. hold the offset fixed in the outer clean region and compare two or three Costas gain settings
2. push the sweep beyond `\\pi/4` and inspect exactly how the coarse estimate aliases
3. replace the deterministic symbol stream with a pulse-shaped stream later, once the symbol-rate model is fully mapped

Useful references for the repo version of this topic:

- `reports/qpsk-frequency-acquisition.md`
- `assets/qpsk-acquisition-range-map.png`
- `notes` or sidecar work in the SDR visual repo when you want the receive-chain explanation instead of the runnable lab
